# Binary Prepayment Baseline

## Objective
This notebook evaluates a binary baseline for prepayment prediction.

We train a neural network with the target defined as:

- `1 = prepay`
- `0 = continue or default`

The goal is to test whether the competing-risk formulation is helping prepayment prediction.

## Dataset

We use the same loan-month dataset and the same temporal split as in the other experiments.

Target encoding in the raw data:

- `0 = continue`
- `1 = prepay`
- `2 = default`



In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

MONTHLY_PATH = "/content/drive/MyDrive/loan_project_ncr/monthly_df_tb3ms_borrower.parquet"

df = pd.read_parquet(MONTHLY_PATH)

print(df.shape)
df.head()

(1656440, 35)


,loan_id,cohort_year,month_since_orig,status_next,tbill_level,tbill_change_3m,apr_tbill_spread,apr_tbill_spread_change_3m,apr_tbill_spread_pct,borrower_apr,...,credit_score_upper_is_missing,dti_is_missing,delinq_7y_is_missing,open_credit_lines_is_missing,revolving_balance_is_missing,bankcard_util_is_missing,available_bankcard_credit_is_missing,income_range_is_missing,employment_status_is_missing,homeowner_is_missing
0,0,2007,1,0.0,0.0420,-0.0053,0.1160,-0.0053,0.734177,0.158,...,0,0,0,0,0,0,0,0,0,0
1,0,2007,2,0.0,0.0389,-0.0072,0.1191,-0.0072,0.753797,0.158,...,0,0,0,0,0,0,0,0,0,0
2,0,2007,3,0.0,0.0390,-0.0092,0.1190,-0.0092,0.753165,0.158,...,0,0,0,0,0,0,0,0,0,0
3,0,2007,4,0.0,0.0327,-0.0093,0.1253,-0.0093,0.793038,0.158,...,0,0,0,0,0,0,0,0,0,0
4,0,2007,5,0.0,0.0300,-0.0089,0.1280,-0.0089,0.810127,0.158,...,0,0,0,0,0,0,0,0,0,0


For this notebook, we define:


In [ ]:
df["target_prepay"] = (df["status_next"] == 1).astype(int)

df["target_prepay"].value_counts()

,count
target_prepay,
0,1643506
1,12934


## Feature Set

We use the same borrower, loan, and macroeconomic features as in the corresponding NCR experiment so that the comparison remains fair.

In [ ]:
features = [
    "month_since_orig",
    "tbill_level",
    "tbill_change_3m",
    "apr_tbill_spread",
    "apr_tbill_spread_change_3m",
    "apr_tbill_spread_pct",
    "borrower_apr",
    "loan_amount_log",
    "prosper_score",
    "credit_score_lower",
    "credit_score_upper",
    "dti",
    "delinq_7y",
    "open_credit_lines",
    "revolving_balance",
    "bankcard_util",
    "available_bankcard_credit",
    "income_range",
    "employment_status",
    "homeowner",
]

In [ ]:
train_df = df[df["cohort_year"] <= 2011]
val_df = df[df["cohort_year"] == 2012]
test_df = df[df["cohort_year"].isin([2013, 2014])]

## Preprocessing

All preprocessing is fit on the training set only.

We apply:
- median imputation
- feature scaling with `StandardScaler`

This keeps the validation pipeline leakage-safe.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_train = train_df[features]
X_val = val_df[features]

y_train = train_df["target_prepay"]
y_val = val_df["target_prepay"]

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## Model

We use the same feedforward neural architecture as in the binary default baseline.

This keeps the binary experiments aligned and makes interpretation easier.

In [ ]:
import torch
import torch.nn as nn

class DefaultMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128,1)
        )

    def forward(self,x):
        return self.net(x)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train).float().to(device)
y_train_t = torch.tensor(y_train.values).float().to(device)

X_val_t = torch.tensor(X_val).float().to(device)
y_val_t = torch.tensor(y_val.values).float().to(device)

In [ ]:
model = DefaultMLP(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

## Training Procedure

The model is trained for 15 epochs.

At the end of each epoch, we generate validation probabilities and compute PR-AUC.

In [ ]:
from sklearn.metrics import average_precision_score

for epoch in range(15):

    model.train()

    optimizer.zero_grad()

    logits = model(X_train_t).squeeze()

    loss = criterion(logits, y_train_t)

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_logits = model(X_val_t).squeeze()

        val_probs = torch.sigmoid(val_logits).cpu().numpy()

        pr_auc = average_precision_score(y_val, val_probs)

    print(epoch, loss.item(), pr_auc)

0 0.6521016359329224 0.008357155559243613
1 0.6293364763259888 0.008479238590520126
2 0.6072759032249451 0.008622422812871185
3 0.5859776139259338 0.008760081712325726
4 0.5654648542404175 0.008887169914533904
5 0.5456515550613403 0.009142731055260029
6 0.5264317989349365 0.009239801246471654
7 0.5078137516975403 0.009270817977392458
8 0.489693820476532 0.009345731030541057
9 0.4720666706562042 0.00938825857631413
10 0.4548732042312622 0.009377814406011263
11 0.4382018744945526 0.009398547294509707
12 0.42179062962532043 0.009431681416358773
13 0.40585073828697205 0.009466765498272224
14 0.3902544677257538 0.009499728598772872


## Results

Validation PR-AUC gradually improves during training and reaches approximately **0.0095**.

## Interpretation

This result is better than the binary default baseline, but still weaker than the competing-risk model.

That suggests prepayment prediction benefits from modeling the interaction between continuing, defaulting, and prepaying, rather than treating prepayment as an isolated binary event.

## Conclusion

The binary prepayment experiment provides a useful diagnostic, but it does not outperform the NCR framework.

This supports keeping competing outcomes in the main modeling approach.